# RawGrad — Convolutional Neural Network From Scratch

This experiment implements and evaluates a small convolutional neural network using RawGrad.

The CNN is built from scratch with NumPy and demonstrates:

- Convolution
- ReLU activation
- Max pooling
- Flattening
- Dense classification
- Softmax probabilities
- Forward propagation
- Backpropagation
- Gradient-based optimization

No deep-learning framework is used for the CNN implementation.

## CNN Architecture

The network follows this pipeline:

Input Image
→ Conv2D
→ ReLU
→ MaxPool2D
→ Flatten
→ Dense
→ Softmax

The convolution layer learns spatial features, pooling reduces spatial
dimensions, and the dense layer performs final classification.

In [11]:
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from rawgrad.activations import ReLU, Softmax
from rawgrad.convolution import Conv2D
from rawgrad.flatten import Flatten
from rawgrad.layers import Dense
from rawgrad.model import Sequential
from rawgrad.pooling import MaxPool2D

In [12]:
np.random.seed(42)

X = np.random.rand(4, 8, 8, 1)

print("Input shape:", X.shape)

Input shape: (4, 8, 8, 1)


In [13]:

model = Sequential([
    Conv2D(2, 3, 1, 1),
    ReLU(),
    MaxPool2D(2, 2),
    Flatten(),
    Dense(18, 10, initialization="xavier"),
    Softmax(),
])

In [14]:
import inspect

print(inspect.signature(Conv2D))
print(inspect.signature(MaxPool2D))

(num_filters, kernel_size, input_channels, stride=1)
(pool_size=2, stride=2)


In [15]:
print("X shape:", X.shape)
print("X dtype:", X.dtype)

X shape: (4, 8, 8, 1)
X dtype: float64


In [16]:
# Convert MNIST data to RawGrad CNN format:
# (batch, height, width, channels)

if X.ndim == 2:
    # (batch, 784) -> (batch, 28, 28, 1)
    X = X.reshape(-1, 28, 28, 1)

elif X.ndim == 3:
    # (batch, 28, 28) -> (batch, 28, 28, 1)
    X = X[..., np.newaxis]

elif X.ndim == 4:
    # If data is channels-first: (batch, channels, height, width)
    if X.shape[1] == 1 and X.shape[3] != 1:
        X = np.transpose(X, (0, 2, 3, 1))

print("CNN input shape:", X.shape)

CNN input shape: (4, 8, 8, 1)


In [17]:
predictions = model.forward(X)

print("Output shape:", predictions.shape)
print("Probability sums:", np.sum(predictions, axis=1))

Output shape: (4, 10)
Probability sums: [1. 1. 1. 1.]


In [18]:
targets = np.zeros((4, 10))

labels = np.array([0, 3, 5, 9])

targets[np.arange(4), labels] = 1.0

print("Target labels:", labels)
print("Target shape:", targets.shape)

Target labels: [0 3 5 9]
Target shape: (4, 10)


In [19]:
from rawgrad.classification_losses import CategoricalCrossEntropy

loss_function = CategoricalCrossEntropy()

loss = loss_function.forward(predictions, targets)
gradient = loss_function.backward()

model.backward(gradient)

print("Loss:", loss)
print("Backward pass: PASS")

Loss: 3.388099017570733
Backward pass: PASS


In [20]:
from rawgrad.optimizers import SGD

optimizer = SGD(learning_rate=0.01)

optimizer.step(model)

print("CNN parameter update: PASS")

CNN parameter update: PASS
